### 1. Importing Libraries

In [28]:
import time
import sqlite3
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Optional
import requests
import pandas as pd

### 2. Configuring Constants

In [29]:
API_URL = "https://www.googleapis.com/books/v1/volumes"
QUERY = "python programming"
TOTAL_BOOKS = 60        # how many books to fetch in total
PAGE_SIZE = 40          # Google Books allows at most 40 per request
DB_PATH = "books.db"


try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_BOOKS_API_KEY")
except Exception:
    API_KEY = None

### 3. Fetching Books from API

In [30]:
def fetch_books_json(query: str, total: int, page_size: int = 40, api_key: Optional[str] = None) -> list:
    """Fetch up to `total` raw book items from Google Books, page by page."""
    session = requests.Session()
    items, start = [], 0

    while len(items) < total:
        params = {
            "q": query,
            "startIndex": start,
            "maxResults": min(page_size, total - len(items)),
            "printType": "books",
        }
        if api_key:
            params["key"] = api_key

        for attempt in range(3):
            response = session.get(API_URL, params=params, timeout=10)
            if response.status_code in (429, 500, 503):      # temporary problems -> retry
                time.sleep(2 ** attempt)
                continue
            response.raise_for_status()                      # other errors -> stop with message
            break
        else:
            raise RuntimeError("API still failing after 3 attempts")

        page = response.json().get("items", [])
        if not page:                                         # no more results
            break
        items.extend(page)
        start += len(page)

    return items

### 4. Parsing JSON Response

In [31]:
@dataclass(frozen=True)
class Book:
    book_id: str
    title: str
    author: str
    publisher: Optional[str]
    year: Optional[int]
    pages: Optional[int]


def parse_books(items: list) -> list:
    books = []
    for item in items:
        info = item.get("volumeInfo", {})
        book_id, title = item.get("id"), (info.get("title") or "").strip()
        if not book_id or not title:
            continue

        date = info.get("publishedDate", "")
        year = int(date[:4]) if date[:4].isdigit() else None

        books.append(Book(
            book_id=book_id,
            title=title,
            author=", ".join(info.get("authors", ["Unknown"])),
            publisher=info.get("publisher"),
            year=year,
            pages=info.get("pageCount") or None,
        ))
    return books

### 5. Initializing SQLite Database

In [32]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS books (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    book_id    TEXT NOT NULL UNIQUE,
    title      TEXT NOT NULL,
    author     TEXT NOT NULL,
    publisher  TEXT,
    year       INTEGER CHECK (year IS NULL OR year BETWEEN 1000 AND 2100),
    pages      INTEGER,
    fetched_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);
CREATE INDEX IF NOT EXISTS idx_books_year ON books(year);
"""

UPSERT = """
INSERT INTO books (book_id, title, author, publisher, year, pages)
VALUES (:book_id, :title, :author, :publisher, :year, :pages)
ON CONFLICT(book_id) DO UPDATE SET
    title = excluded.title, author = excluded.author, publisher = excluded.publisher,
    year = excluded.year, pages = excluded.pages, fetched_at = CURRENT_TIMESTAMP
"""


@contextmanager
def get_connection(path: str):
    conn = sqlite3.connect(path)
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def save_books(conn, books: list) -> int:
    conn.executescript(SCHEMA)
    conn.executemany(UPSERT, [vars(b) for b in books])
    return len(books)

### 6. Running Data Pipeline

In [33]:
raw_items = fetch_books_json(QUERY, TOTAL_BOOKS, PAGE_SIZE, API_KEY)
print(f"Fetched {len(raw_items)} raw records")

books = parse_books(raw_items)
print(f"{len(books)} valid books after validation")

with get_connection(DB_PATH) as conn:
    saved = save_books(conn, books)
print(f"Saved/updated {saved} books in {DB_PATH}")

Fetched 60 raw records
60 valid books after validation
Saved/updated 60 books in books.db


### 7. Querying and Displaying Results

In [34]:
with get_connection(DB_PATH) as conn:
    df = pd.read_sql_query(
        "SELECT title, author, publisher, year, pages FROM books "
        "ORDER BY year IS NULL, year DESC, title",
        conn,
    )
df

,title,author,publisher,year,pages
0,Machine Learning and Python Programming,"Dr. M. Elavarasi, Dr. Lipika Mandal, Dr. Satee...",BR Publications,2026.0,279.0
1,Python Programming for Machine Learning,"Dr.A.Kalpana, Mrs.S.Sree Priya, Dr.K.Sivakami",Leilani Katie Publication,2024.0,201.0
2,PYTHON PROGRAMMING,"Dr.M Sirish Kumar, G.Radhika, A.Basi Reddy",RAJA SURESH,2023.0,192.0
3,Python Programming Recipes for IoT Applications,"Jivan S. Parab, Madhusudan Ganuji Lanjewar, Ma...",Springer Nature,2023.0,206.0
4,Basic And Advance : Python Programming,Agya Ram Verma,New India Publishing Agency,2022.0,13.0
5,Python Programming Guide for GCSE Computer Sci...,CGP Books,CGP Ltd,2022.0,132.0
6,Learn Python,Damon Parker,Damon Parker,2021.0,141.0
7,Learn Python Programming,Jason Crash,None,2021.0,190.0
8,Python Coding and Programming,Michael Learn,Michael Learn,2021.0,222.0
9,Python Programming,"Vijay Kumar Sharma, Vimal Kumar, Swati Sharma,...",CRC Press,2021.0,403.0
